In [ ]:
!pip install -q torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.9 MB/s eta 0:00:00


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config B ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/grpo-checkpoint-epoch-0"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...
✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 54.0%  (27/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       54.0%  ███████████████████████████


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config B ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/grpo-checkpoint-epoch-1"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...
✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 60.0%  (30/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       60.0%  ██████████████████████████████


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config B ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/grpo-checkpoint-epoch-2"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...
✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 44.0%  (22/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       44.0%  ██████████████████████


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config C ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/latest--grpo-checkpoint-epoch-0"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...
✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 44.0%  (22/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       44.0%  ██████████████████████


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config  C ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/latest--grpo-checkpoint-epoch-1"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...
✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 48.0%  (24/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       48.0%  ████████████████████████


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip uninstall -y torchao
!pip install -U "torchao>=0.16.0"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.3 MB/s eta 0:00:00


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/check1/grpo-checkpoint-epoch-0"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 48.0%  (24/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       48.0%  ████████████████████████


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/check1/grpo-checkpoint-epoch-1"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...
✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 38.0%  (19/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       38.0%  ███████████████████


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# ── Config ────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
SFT_ADAPTER    = "/content/drive/MyDrive/backup_drive1/MyDrive/My_Qwen_Math_Training/checkpoint-epoch-5"
GRPO_ADAPTER   = "/content/drive/MyDrive/check1/grpo-checkpoint-epoch-2"  # ← update to your best checkpoint
EVAL_SAMPLES   = 50

# ── Helpers ───────────────────────────────────────────────────
def extract_number(text):
    text = text.replace(',', '')
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def format_prompt(question):
    return (
        'You are a math reasoning assistant. '
        'Solve the problem step by step, then give your final answer as a number.\n\n'
        f'Problem: {question}\n\nSolution:'
    )

def evaluate(model, tokenizer, data, label):
    model.eval()
    correct = 0
    for example in data:
        prompt = format_prompt(example['question'])
        gold   = extract_number(example['answer'].split('####')[-1].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = extract_number(completion)
        if pred and gold and pred == gold:
            correct += 1
    acc = 100 * correct / len(data)
    print(f"  {label:<25} → {acc:.1f}%  ({correct}/{len(data)})")
    return acc

# ── Load eval data ────────────────────────────────────────────
print('📥 Loading GSM8K test set...')
gsm8k     = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} samples ready\n')

results = {}

# # ── 1. Baseline: raw Qwen2.5-1.5B ────────────────────────────
print('📊 Evaluating baseline (no fine-tuning)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
base_model.config.pad_token_id = tokenizer.eos_token_id
results['Baseline Qwen2.5-1.5B'] = evaluate(base_model, tokenizer, eval_data, 'Baseline Qwen2.5-1.5B')
del base_model
torch.cuda.empty_cache()

# ── 2. SFT model ──────────────────────────────────────────────
print('📊 Evaluating SFT model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER, is_trainable=False)
# sft_model.config.pad_token_id = tokenizer.eos_token_id
# results['SFT'] = evaluate(sft_model, tokenizer, eval_data, 'SFT (NuminaMath-CoT)')
# del sft_model, base_model
# torch.cuda.empty_cache()

# ── 3. GRPO model ─────────────────────────────────────────────
print('📊 Evaluating GRPO model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
grpo_model = PeftModel.from_pretrained(base_model, GRPO_ADAPTER, is_trainable=False)
grpo_model.config.pad_token_id = tokenizer.eos_token_id
results['GRPO'] = evaluate(grpo_model, tokenizer, eval_data, 'GRPO (GSM8K)')
del grpo_model, base_model
torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
print('\n' + '='*45)
print('  GSM8K Accuracy — 50 test samples')
print('='*45)
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f"  {name:<25} {acc:5.1f}%  {bar}")
print('='*45)

📥 Loading GSM8K test set...
✅ 50 samples ready

📊 Evaluating baseline (no fine-tuning)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Baseline Qwen2.5-1.5B     → 34.0%  (17/50)
📊 Evaluating SFT model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📊 Evaluating GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  GRPO (GSM8K)              → 32.0%  (16/50)

  GSM8K Accuracy — 50 test samples
  Baseline Qwen2.5-1.5B      34.0%  █████████████████
  GRPO                       32.0%  ████████████████
